# 00. Build every result table and figure

This notebook runs `scripts/make_results.sh` step by step, one cell per step and in the same order: it turns the
saved results into the summary CSVs and figures in `reports/` and the result tables in `reports/tables/`,
`reports/causal/` and `reports/attack_views/`. Together they hold every table and figure of the main study
(Tables II to VII, Figures 1 and 2) and of the supplement (Tables S-II to S-XLIII, Figures S1 to S18); `RESULTS.md`
says which file holds which result.

**Inputs:** `runs/`, `runs_consistent/`, `runs_ivisvar/report/`, `data/processed/<dataset>/meta.json`, and,
optionally, the LoRA / zero-shot foundation-model runs in the folder named by the environment variable
`FM_ADAPT_RUNS`. The whole build takes about two minutes. The other notebooks make the same results one topic at
a time.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

The shell script runs each step in a new Python process. Here all steps share one process, so two things are set
as the script has them: figures are drawn into files only (the Agg backend; they are shown below from their PNG
files), and matplotlib's settings are reset before each step, because `make_report` sets its house style globally
and a later step would otherwise inherit it (and write different figure bytes).

In [ ]:
import os
import matplotlib

matplotlib.use("agg")


def fresh_matplotlib():
    """matplotlib's settings as a new Python process has them (the style an earlier step set is dropped)."""
    import matplotlib.pyplot as plt

    plt.close("all")
    matplotlib.rc_file_defaults()


(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)   # make_results.sh: mkdir -p reports/tables

**LoRA and zero-shot foundation models** (`report/fm_adapt.py`): Table S-XXVIII
(`reports/table17_fm_adapt.csv`) and their points in the Pareto figure (`reports/pareto_extra_fm_adapt.csv`, read
by the next step for Figure S7). Runs only when `FM_ADAPT_RUNS` is set, as in the script.

In [ ]:
FM_ADAPT_RUNS = os.environ.get("FM_ADAPT_RUNS")
if FM_ADAPT_RUNS:
    from fraudadv.report import fm_adapt

    fresh_matplotlib()
    fm_adapt.main(Path(os.environ.get("FRAUDADV_RUNS", "runs")), fm_adapt.RL_ROOT / "results_adapt", Path("reports"),
                  Path(FM_ADAPT_RUNS))
else:
    print("FM_ADAPT_RUNS is not set: skipping Table S-XXVIII and the LoRA / zero-shot Pareto points")

**Summary CSVs and figures** (`report/make_report.py: main`): every summary in `reports/` (clean performance
`table1_clean.csv`, the grid summaries, thickness, explanations, statistics, defenses, certification, ranking
metrics, history injection, halves, reruns) and the report figures: Figure 1 and Figures S1-S3, S5-S7.

In [ ]:
from fraudadv.report import make_report

fresh_matplotlib()
make_report.main()

**Controlled ablations** (`causal/analyze.py`, then `causal/report.py`): Table VI and Tables S-XXXIV to
S-XXXVI (`reports/causal/contrasts.csv`, `factorial_lpm.csv`, `variants_clean.csv`, `factorial_effects.csv`) and
Figures S13-S14.

In [ ]:
from fraudadv.causal import analyze

fresh_matplotlib()
analyze.main()

**Attack views** (`report/attack_views.py figures`, which calls `report/attack_figures.py: main`): Figure 2,
Figures S15-S18, Table S-XXXVII (`reports/attack_views/attack_examples*.csv`) and the numbers quoted with Figure 2
(`reports/attack_views/attack_view_numbers.csv`).

In [ ]:
from fraudadv.report import attack_figures

fresh_matplotlib()
attack_figures.main()

**SHAP and attribution figures** (`report/shap_figures.py`): Figures S8-S12, with their numbers in
`reports/shap/`.

In [ ]:
from fraudadv.report import shap_figures

fresh_matplotlib()
shap_figures.main()

**Result tables** (`report/tables.py`): the tables that need more than the summary CSVs, in `reports/tables/`:
Table II (`datasets.csv`), Table V (`foundation_models.csv`), Table S-II (`hpo_trials.csv`), Tables S-V to S-X
(`grid_per_dataset.csv`), Table S-XXI (`budget_check.csv`), Table S-XXX (`robust_auc_mean.csv`), Table S-XXXIII
(`drivers.csv`) and the top rows of Table S-XL (`halves_by_dataset.csv`).

In [ ]:
from fraudadv.report import tables

tables.main()

**ivis variants** (`scripts/analysis/ivis_variants_paired.py`, run as the script): the Holm-adjusted paired
comparisons of Table S-XXVI (`runs_ivisvar/report/paired_holm.csv`). Tables S-XXV and S-XXVII are
`runs_ivisvar/report/clean.csv`, `worst_case.csv` and `geometry.csv`.

In [ ]:
import runpy

runpy.run_path(str(ROOT / "scripts" / "analysis" / "ivis_variants_paired.py"), run_name="__main__");

**Impossible records** (`report/realizable_share.py`): `reports/analysis/consistent/realizable.csv`, the shares
of evasions no real transaction could produce (Section VI-C numbers, input of Table S-XIII).

In [ ]:
from fraudadv.report import realizable_share

realizable_share.main()

**Consistent attacker** (`report/consistent_tables.py`): Table III (`reports/tables/worst_case_three_attackers.csv`
and `worst_case_rank.csv`), Tables S-XIII to S-XX and S-XXIX (the other CSVs it writes to `reports/tables/`) and
Figure S4 (`reports/fig_transfer_consistent.png`).

In [ ]:
from fraudadv.report import consistent_tables

fresh_matplotlib()
consistent_tables.main()

## A few results

Table III: worst-case attack success rate at $\epsilon = 0.25$ under the three attackers (U unconstrained,
C constrained, K consistent), one column per dataset and attacker, and each model's mean normalized rank.

In [ ]:
import pandas as pd

W = pd.read_csv(ROOT / "reports" / "tables" / "worst_case_three_attackers.csv")
table3 = W.pivot_table(index="model", columns=["dataset", "attacker"], values="asr").round(3)
rank = pd.read_csv(ROOT / "reports" / "tables" / "worst_case_rank.csv").set_index("model")["mean_normalized_rank"]
table3[("mean normalized rank", "")] = rank.round(2)
table3

Figure 1: transfer rates of constrained evasions between models.

In [ ]:
nb.show_figure("fig_transfer_constrained")

## Check against the reference build

Every file this notebook wrote under `reports/` is compared byte for byte with the same path in the reference build
named by `FRAUDADV_REFERENCE` (`.gz` files decompressed).

In [ ]:
files = sorted(p.relative_to(ROOT).as_posix() for p in (ROOT / "reports").rglob("*") if p.is_file())
print(len(files), "files to compare")
ok = nb.check(files)